# SmolVLA × LIBERO-Goal — Phase 3 SFT (Colab Pro+, A100)

Fine-tune `lerobot/smolvla_base` on `HuggingFaceVLA/libero` filtered to the **Goal suite**.
Comparison target: **85.0%** (`results/baseline_libero_goal.json` — the official checkpoint under our validated protocol).

**Read before running:**
- 🖥️ Runtime: **A100 + High-RAM, background execution ON**. This notebook *trains only* — all real evals run on **L4** via `baseline_eval_colab.ipynb` (sim is CPU-bound; A100 units are wasted on it).
- 🚦 **Gates, in order:** (1) the versions cell prints a GPU; (2) dataset recon finds ~500 Goal episodes across 10 tasks; (3) the **SMOKE run** passes loss-down → checkpoint saved → `RESUME=True` works → eval-load smoke passes — **only then** flip `SMOKE=False` for the 20k full run.
- 💰 Note the **compute-unit meter** before/after the smoke run — that calibrates the full-run cost.
- 💾 Checkpoints + logs go to **Drive**: a disconnect loses at most `SAVE_FREQ` steps; resume with `RESUME=True`.
- 🌐 Prefer the **Colab web UI** over VS Code for long runs (High-RAM toggle + background execution are first-class there).
- ⏱️ Full run: 20k steps ≈ **4–8 h** on A100 (the smoke run gives the real rate).
- Every cell is **re-runnable**; re-run from the top after a reconnect.

## 1. GPU check

In [ ]:
!nvidia-smi
import sys; print('python', sys.version)
# Expect an A100 here. If nvidia-smi is 'command not found', you got a CPU
# runtime -- Runtime > Change runtime type > A100 GPU + High-RAM, then restart.

## 2. Install pinned environment

Same stack as the eval notebook (lerobot[smolvla,libero] + render libs — the
libero package is also our ground truth for Goal-suite task strings, and the
eval-load smoke below needs the env).

In [ ]:
!sudo apt-get -qq update
!sudo apt-get -qq install -y ffmpeg libegl1-mesa-dev libgl1-mesa-glx libosmesa6-dev libglfw3 > /dev/null
print('apt deps installed')

In [ ]:
%pip install -q "lerobot[smolvla,libero]" 2>&1 | tail -n 5
print('lerobot install step done')

In [ ]:
# Record EXACT resolved versions + HARD GPU GATE (training on CPU is pointless).
import importlib, platform, torch
def v(m):
    try: return importlib.import_module(m).__version__
    except Exception as e: return 'ERR ' + str(e)
print('python      ', platform.python_version())
print('lerobot     ', v('lerobot'))
print('torch       ', torch.__version__, '| cuda', torch.version.cuda, '| avail', torch.cuda.is_available())
print('transformers', v('transformers'))
if not torch.cuda.is_available():
    raise SystemExit('NO GPU on this runtime -- pick A100 + High-RAM before going any further.')
print('GPU:', torch.cuda.get_device_name(0), '|', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 3. Env vars — HF cache on LOCAL disk, never on Drive

The dataset cache is thousands of files; Drive IO would crawl. Local runtime
disk is ephemeral but the download (only the Goal-episode shards) is fast to redo.

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'                  # for the eval-load smoke
os.environ['HF_HOME'] = '/content/hf_cache'      # LOCAL disk -- never Drive
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('MUJOCO_GL =', os.environ['MUJOCO_GL'], '| HF_HOME =', os.environ['HF_HOME'])

## 4. Mount Google Drive (checkpoints, logs, records persist here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
OUT_BASE = '/content/drive/MyDrive/VLATune/sft'
pathlib.Path(OUT_BASE).mkdir(parents=True, exist_ok=True)
print('outputs ->', OUT_BASE)

## 5. Logins (optional, never persisted)

- **HF**: `HuggingFaceVLA/libero` and `lerobot/smolvla_base` are public — login only
  if you hit rate limits. **Never paste a token into a saved cell.**
- **W&B**: only needed if you set `WANDB=True` in the train cell. Run
  `import wandb; wandb.login()` in a *throwaway* cell (interactive prompt; key is
  not stored in the notebook).

In [ ]:
# Optional, non-persisted:
# from huggingface_hub import login; from getpass import getpass
# login(getpass('HF token (blank to skip): ') or None)
# import wandb; wandb.login()
print('skip unless you hit rate limits / want W&B curves')

## 6. Reconcile `lerobot-train` flags against the INSTALLED version

Same rule as Phase 2: the installed `--help` is the source of truth. The flag
names below come from the plan + SmolVLA docs and are **unverified until this
cell runs**. (Remember the draccus lesson: list args need brackets, `[0,1,2]`.)

In [ ]:
help_txt = get_ipython().getoutput('lerobot-train --help')
help_txt = '\n'.join(help_txt)
for f in ['policy.path', 'dataset.repo_id', 'dataset.episodes', 'output_dir', 'job_name',
          'steps', 'batch_size', 'save_freq', 'log_freq', 'num_workers',
          'wandb.enable', 'resume', 'config_path']:
    print(('FOUND   ' if f in help_txt else 'MISSING '), '--' + f)
print()
print('--- if MISSING: the installed name wins; adjust the train cell below ---')
print('--- (policy.path may show MISSING like in eval -- it is a nested arg and works) ---')

## 7. Dataset recon — metadata only (GATE: ~500 Goal episodes, 10 tasks)

No 34GB pull — this loads only the dataset's `meta/` files. Ground-truth Goal
task strings come from the installed `libero` package itself (no hardcoding),
then we match them against per-episode task strings in `HuggingFaceVLA/libero`.

Writes `dataset_goal_episodes.json` to Drive — **copy it into the repo as
`results/dataset_goal_episodes.json`** so the episode list is versioned.
If it finds 0 matches it prints the dataset's actual task strings so the
matching can be fixed by inspection — do NOT train until this gate passes.

In [ ]:
import os, re, json
from collections import Counter

# 1) ground-truth Goal task strings from the libero package
from libero.libero import benchmark as lb_benchmark
suite = lb_benchmark.get_benchmark_dict()['libero_goal']()
goal_langs = [suite.get_task(i).language for i in range(suite.n_tasks)]
norm = lambda s: re.sub(r'[^a-z0-9 ]', '', str(s).lower()).strip()
goal_set = {norm(l) for l in goal_langs}
print('libero_goal ground-truth task strings (from the libero package):')
for l in goal_langs: print('  -', l)

# 2) dataset metadata only (defensive import across lerobot layouts)
try:
    from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata
except ImportError:
    from lerobot.common.datasets.lerobot_dataset import LeRobotDatasetMetadata
REPO = 'HuggingFaceVLA/libero'
meta = LeRobotDatasetMetadata(REPO)
print()
print('repo:', REPO, '| episodes:', meta.total_episodes, '| frames:', meta.total_frames, '| fps:', meta.fps)
print('features:')
for k, val in meta.features.items():
    dt = val.get('dtype') if isinstance(val, dict) else None
    sh = val.get('shape') if isinstance(val, dict) else None
    print('  ' + k.ljust(38), 'dtype=', dt, 'shape=', sh)
for k in ['observation.images.image', 'observation.images.image2', 'observation.state', 'action']:
    print(('OK      ' if k in meta.features else 'MISSING ') + k)

# 3) per-episode task strings -> Goal episode indices (layout-agnostic)
def iter_episodes(eps):
    try:
        import pandas as pd
        if isinstance(eps, pd.DataFrame):
            for idx, row in eps.iterrows():
                yield int(row['episode_index']) if 'episode_index' in row else int(idx), row['tasks']
            return
    except ImportError:
        pass
    if isinstance(eps, dict):
        for idx, info in eps.items():
            yield int(idx), info.get('tasks', info.get('task', []))
        return
    for idx, info in enumerate(eps):
        yield idx, info.get('tasks', info.get('task', []))

goal_eps, per_task, unmatched = [], Counter(), Counter()
for idx, tasks in iter_episodes(meta.episodes):
    if tasks is None: tasks = []
    if isinstance(tasks, str): tasks = [tasks]
    tasks = [str(t) for t in tasks]
    hit = [t for t in tasks if norm(t) in goal_set]
    if hit:
        goal_eps.append(idx); per_task[hit[0]] += 1
    else:
        for t in tasks: unmatched[t] += 1

print()
print('Goal episodes found:', len(goal_eps), '(expect ~500 = 10 tasks x ~50 demos)')
for t, n in sorted(per_task.items()): print('  ' + str(n).rjust(4) + '  ' + t)
if len(goal_eps) == 0:
    print()
    print('NO MATCHES -- dataset task strings differ from libero language strings.')
    print('Most common dataset task strings (fix the matching, then re-run):')
    for t, n in unmatched.most_common(60): print('  ' + str(n).rjust(4) + '  ' + t)
    raise SystemExit('GATE FAILED: fix task-string matching before training.')
if len(per_task) != 10:
    print('WARNING: matched', len(per_task), '/10 Goal tasks -- inspect before trusting this list.')

rec = {'repo': REPO, 'suite': 'libero_goal', 'n_episodes': len(goal_eps),
       'per_task_counts': dict(per_task), 'episodes': sorted(goal_eps)}
EP_FILE = os.path.join(OUT_BASE, 'dataset_goal_episodes.json')
json.dump(rec, open(EP_FILE, 'w'), indent=2)
print()
print('wrote', EP_FILE)
print('-> copy into the repo as results/dataset_goal_episodes.json')

## 8. Train — full recipe (VALIDATED 2026-06-15 on A100-80GB)

See `docs/phase3_sft_runbook.md` for the full story. Five fixes the smoke caught,
all baked into the cell below:
1. **`--policy.push_to_hub=false`** — else aborts wanting `policy.repo_id`.
2. **`HuggingFaceVLA/libero` episode→file metadata is BROKEN** (every revision) —
   episode-filtered download fetches wrong files. Fix: download the FULL
   `data/`+`meta/` (~35 GB; images embedded in parquet, no videos) ONCE, set
   `HF_LEROBOT_HOME` to point lerobot's cache at it, then the `episode_index`
   filter selects the 428 goal episodes correctly.
3. **`--rename_map` image→camera1, image2→camera2** — `smolvla_base` expects 3
   cameras (camera1/2/3); LIBERO has 2. After rename, `{camera1,camera2}` ⊆
   `{camera1,2,3}` passes the check; runtime skips the missing camera3.
4. **Eval of this checkpoint needs the SAME `--rename_map`** (it keeps camera1/2/3).
5. **Pre-create `~/.libero/config.yaml`** so eval subprocesses don't EOF on the
   first-import `input()` prompt.

Measured: batch 64 fits A100-80GB (no OOM), ~2 steps/s → 20k ≈ **2.8 h**,
loss decreases. `save_freq=2000` → Drive (preemption-safe; re-run to auto-resume).
Run on **A100 + High-RAM, background execution ON**. Eval the `last` checkpoint
separately on **L4** via `baseline_eval_colab.ipynb` vs the **85.0%** baseline.

In [ ]:
# VALIDATED full SFT recipe (2026-06-15). Background-launch; poll with the next cell.
# PREREQ (run once, ~35GB, ~5-40 min): download the FULL dataset because the
# episode->file metadata is broken (see runbook). Then HF_LEROBOT_HOME points
# lerobot's cache at it so the episode_index filter selects the 428 goal episodes.
import json, os, shutil, subprocess, time, textwrap

OUT_BASE = '/content/drive/MyDrive/VLATune/sft'
HF_LEROBOT_HOME = '/content/hf_cache'            # lerobot cache = here/hub
os.environ['HF_HOME'] = HF_LEROBOT_HOME

# --- step A: full dataset download (idempotent; skips if already complete) ---
DL = '/content/dl_libero.py'
open(DL, 'w').write(textwrap.dedent('''
    from huggingface_hub import snapshot_download
    p = snapshot_download("HuggingFaceVLA/libero", repo_type="dataset", revision="v3.0",
                          allow_patterns=["data/*", "meta/*"])
    print("DONE", p, flush=True)
'''))
print('Downloading full dataset (idempotent)...')
subprocess.run(['python', DL], env=dict(os.environ, HF_HOME=HF_LEROBOT_HOME), check=True)

# --- step B: launch the 20k SFT run ---
SMOKE = False
RUN_NAME = 'smolvla_sft_goal_smoke' if SMOKE else 'smolvla_sft_goal_run1'
STEPS = 100 if SMOKE else 20000
BATCH_SIZE, SAVE_FREQ, LOG_FREQ, NUM_WORKERS = 64, (50 if SMOKE else 2000), (25 if SMOKE else 200), 4
RENAME_MAP = '{"observation.images.image":"observation.images.camera1","observation.images.image2":"observation.images.camera2"}'
goal_eps = json.load(open(os.path.join(OUT_BASE, 'dataset_goal_episodes.json')))['episodes']
EPISODES = '[' + ','.join(str(e) for e in goal_eps) + ']'
OUTPUT_DIR = os.path.join(OUT_BASE, RUN_NAME)
RESUME = os.path.isdir(os.path.join(OUTPUT_DIR, 'checkpoints'))
if RESUME:
    cfg = os.path.join(OUTPUT_DIR, 'checkpoints', 'last', 'pretrained_model', 'train_config.json')
    cmd = ['lerobot-train', '--config_path=' + cfg, '--resume=true']
else:
    cmd = ['lerobot-train', '--policy.path=lerobot/smolvla_base', '--policy.push_to_hub=false',
           '--rename_map=' + RENAME_MAP,
           '--dataset.repo_id=HuggingFaceVLA/libero', '--dataset.episodes=' + EPISODES,
           '--output_dir=' + OUTPUT_DIR, '--job_name=' + RUN_NAME, '--steps=' + str(STEPS),
           '--batch_size=' + str(BATCH_SIZE), '--save_freq=' + str(SAVE_FREQ),
           '--log_freq=' + str(LOG_FREQ), '--num_workers=' + str(NUM_WORKERS), '--wandb.enable=false']
env = dict(os.environ, HF_LEROBOT_HOME=HF_LEROBOT_HOME)
LOG = '/content/' + RUN_NAME + '_console.log'
proc = subprocess.Popen(cmd, stdout=open(LOG, 'a'), stderr=subprocess.STDOUT, text=True, env=env)
open('/content/run_pid.txt', 'w').write(str(proc.pid))
open('/content/run_start.txt', 'w').write(str(time.time()))
open('/content/run_name.txt', 'w').write(RUN_NAME)
print('LAUNCHED', ('RESUME ' if RESUME else ''), 'pid', proc.pid, '| steps', STEPS,
      '| batch', BATCH_SIZE, '| goal episodes', len(goal_eps), '| log', LOG)

## 9. Checkpoint eval-load smoke (~3 min, 1 episode)

Proves the trained checkpoint loads in `lerobot-eval` — the same path the real
L4 evals will use. **The episode's success/failure is irrelevant** for a
500-step smoke checkpoint; only `exit code: 0` matters here.

In [ ]:
import os, subprocess
CKPT = os.path.join(OUTPUT_DIR, 'checkpoints', 'last', 'pretrained_model')
SMOKE_OUT = os.path.join(OUT_BASE, RUN_NAME + '_loadsmoke')
cmd = ['lerobot-eval',
       '--policy.path=' + CKPT,
       '--env.type=libero',
       '--env.task=libero_goal',
       '--env.control_mode=relative',
       '--env.max_parallel_tasks=1',
       '--policy.n_action_steps=10',
       '--eval.batch_size=1',
       '--eval.n_episodes=1',
       '--env.task_ids=[0]',
       '--output_dir=' + SMOKE_OUT]
print('RUNNING:')
for a in cmd: print('   ', a)
r = subprocess.run(cmd, capture_output=True, text=True)
tail = (r.stdout + r.stderr).splitlines()[-25:]
print('\n'.join(tail))
print()
print('exit code:', r.returncode)
print('PASS -- checkpoint loads in lerobot-eval' if r.returncode == 0
      else 'FAIL -- fix the checkpoint/load path BEFORE the 20k run')

## 10. Record run summary to Drive

In [ ]:
import json, os, datetime
summary = {
    'run_name': RUN_NAME, 'smoke': SMOKE, 'resumed': RESUME,
    'policy_base': 'lerobot/smolvla_base',
    'dataset': 'HuggingFaceVLA/libero (episodes filtered to libero_goal)',
    'n_goal_episodes': len(goal_eps),
    'steps': STEPS, 'batch_size': BATCH_SIZE, 'save_freq': SAVE_FREQ,
    'num_workers': NUM_WORKERS, 'wandb': WANDB,
    'wall_s': round(TRAIN_WALL_S, 1),
    'output_dir': OUTPUT_DIR,
    'checkpoint_for_eval': os.path.join(OUTPUT_DIR, 'checkpoints', 'last', 'pretrained_model'),
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'baseline_to_beat': '85.0% (results/baseline_libero_goal.json)',
}
p = os.path.join(OUT_BASE, RUN_NAME + '_summary.json')
json.dump(summary, open(p, 'w'), indent=2)
print('wrote', p)
print(json.dumps(summary, indent=2))

## 11. After the full run + troubleshooting

**Eval the checkpoint (separate session, L4 — not here):** open
`baseline_eval_colab.ipynb`, switch to an **L4** runtime, set
`POLICY = '<Drive>/VLATune/sft/smolvla_sft_goal_run1/checkpoints/last/pretrained_model'`
and run the standard 100-episode protocol. Compare against **85.0%**.

**Copy back into the repo afterwards:** `dataset_goal_episodes.json`,
`smolvla_sft_goal_run1_summary.json`, and the eval's `eval_info.json` →
`results/`.

| Symptom | Fix |
|---|---|
| CUDA out of memory (clean traceback) | `BATCH_SIZE = 32`, re-run |
| exit `-9` at any point | host-RAM OOM → lower `NUM_WORKERS`, confirm High-RAM |
| `flag not recognized` | reconcile with cell 6 — the installed name wins |
| loss flat / NaN in smoke | check episode list (cell 7 gate), keep policy defaults |
| dataset download enormous | `--dataset.episodes` should pull only Goal shards; if it pulls all 34GB, note it — Colab disk still fits |
| disconnect mid-run | re-run from top with `RESUME = True` (loses ≤ `SAVE_FREQ` steps) |
| Drive writes stall training | move `OUTPUT_DIR` to `/content/...` and copy `checkpoints/last` to Drive periodically |

Expectation: landing **within a few points of 85%** validates the pipeline —
the official checkpoint trained on more data. The headline result is Phase 4's
SFT→RL delta.